# 02 — Data Cleaning
## RCT World Explorer

This notebook takes the raw AEA registry data (`aea_registry_clean.csv`) and produces a fully cleaned, analysis-ready dataset (`rct_cleaned_data.csv`).

### What this notebook does
1. Fixes raw file issues (multiline rows, encoding artifacts)
2. Parses the `Country names` column — the most complex step
3. Classifies each entry: Private / Online / Single country / Multi-country / Multi-region
4. Handles partial country names (e.g. `Congo (Democratic Republic of the)`)
5. Standardises country spellings via a lookup dictionary
6. Maps every country to `continent` and `continent_region` via a fixed lookup (ignoring bracket content for this — see rationale below)
7. Extracts `region_detail` from bracket content where it is a sub-national location
8. Cleans `Keywords` (strips double-escaping from CSV serialisation)
9. Creates `RCT_ID_duplicates` — a unique row-level identifier
10. Validates and saves

### Key design decision: country → continent via lookup, not bracket content
The `Country names` column stores bracket content inconsistently. The same country can appear as `Afghanistan (South Asia)` or `Afghanistan (Kabul)` — the former gives correct continent info, the latter gives a city. Using bracket content to assign continent therefore produces inconsistent mappings across 48+ countries. The correct approach is a fixed `country → continent` lookup dictionary that ignores bracket content entirely.

## Step 0 — Imports and paths

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from collections import Counter

RAW_PATH  = Path("../data/raw/aea_registry_clean.csv")
OUT_PATH  = Path("../data/processed/rct_cleaned_data.csv")
COMP_PATH = Path("../data/processed/comparison_original_vs_cleaned.csv")

print(f"Raw file exists: {RAW_PATH.exists()}")

Raw file exists: True


## Step 1 — Load raw data and inspect

Before touching anything, we look at the raw data to understand its structure and spot obvious issues.

In [2]:
df_raw = pd.read_csv(RAW_PATH)

print(f"Shape: {df_raw.shape}")
print(f"\nColumns: {df_raw.columns.tolist()}")
print(f"\nDtypes:\n{df_raw.dtypes}")
print(f"\nNull counts:\n{df_raw.isnull().sum()}")
print(f"\nFirst 3 rows:")
df_raw.head(3)

Shape: (6802, 50)

Columns: ['Title', 'Url', 'Last update date', 'Published at', 'First registered on', 'RCT_ID', 'DOI Number', 'Primary Investigator', 'Status', 'Start date', 'End date', 'Keywords', 'Country names', 'Other Primary Investigators', 'Jel code', 'Secondary IDs', 'Abstract', 'External Links', 'Sponsors', 'Partners', 'Intervention start date', 'Intervention end date', 'Intervention', 'Primary outcome end points', 'Primary outcome explanation', 'Secondary outcome end points', 'Secondary outcome explanation', 'Experimental design', 'Experimental design details', 'Randomization method', 'Randomization unit', 'Sample size number clusters', 'Sample size number observations', 'Sample size number arms', 'Minimum effect size', 'IRB', 'Analysis Plan Documents', 'Intervention completion date', 'Data collection completion', 'Data collection completion date', 'Number of clusters', 'Attrition correlated', 'Total number of observations', 'Treatment arms', 'Public data', 'Public data url'

,Title,Url,Last update date,Published at,First registered on,RCT_ID,DOI Number,Primary Investigator,Status,Start date,...,Number of clusters,Attrition correlated,Total number of observations,Treatment arms,Public data,Public data url,Program files,Program files url,Post trial documents csv,Relevant papers for csv
0,Voter Pessimism and Electoral Accountability: ...,http://www.socialscienceregistry.org/trials/5,2017-05-02,2017-05-02 16:17:17 -0400,2013-05-21,AEARCTR-0000005,10.1257/rct.5-8.0,Kelly Zhang kwzhang@mit.edu,completed,2013-01-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Community Based Strategies to Reduce Maternal ...,http://www.socialscienceregistry.org/trials/6,2016-09-07,2016-09-07 17:44:46 -0400,2013-05-26,AEARCTR-0000006,10.1257/rct.6-2.0,Jessica Leight j.leight@cgiar.org,completed,2012-02-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,An Evaluation of Continuous Comprehensive Eval...,http://www.socialscienceregistry.org/trials/8,2013-06-15,2013-06-15 09:00:16 -0400,2013-06-15,AEARCTR-0000008,10.1257/rct.8-1.0,Esther Duflo eduflo@mit.edu,on_going,2011-05-19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 2 — Fix raw file issues

The AEA API returns data that, when serialised to CSV and reloaded, can contain:
- **Multiline rows**: newline characters embedded inside field values (common in `Title` and `Country names`)
- **Encoding artifacts**: stray tab characters, non-breaking spaces
- **RCT_ID format issues**: any rows where the ID doesn't match `AEARCTR-XXXXXXX`

We fix all of these before any parsing begins.

In [4]:
df = df_raw.copy()

# Strip leading/trailing whitespace from all string columns
for col in df.select_dtypes(include='object').columns:
    try:
        df[col] = df[col].str.strip()
    except AttributeError:
        pass

# Replace embedded newlines and tabs in key columns
for col in ['Title', 'Country names', 'RCT_ID']:
    df[col] = df[col].str.replace(r'[\n\t]+', ' ', regex=True).str.strip()

# Validate RCT_ID format
valid_id = df['RCT_ID'].str.match(r'^AEARCTR-\d+$', na=False)
print(f"RCT_IDs with unexpected format: {(~valid_id).sum()}")
if (~valid_id).sum() > 0:
    print(df[~valid_id][['RCT_ID', 'Country names']].head(10))

# Check for any remaining multiline content
for col in ['Title', 'Country names', 'RCT_ID']:
    n = df[col].str.contains('\n', na=False).sum()
    print(f"  {col}: {n} rows still contain newlines")

print(f"\nShape after cleaning: {df.shape}")

RCT_IDs with unexpected format: 0
  Title: 0 rows still contain newlines
  Country names: 0 rows still contain newlines
  RCT_ID: 0 rows still contain newlines

Shape after cleaning: (6802, 50)


## Step 3 — Parse the `Country names` column

This is the most complex step. The `Country names` column stores everything in one string with the pattern:

```
CountryName (bracket_content); CountryName2 (bracket_content2)
```

Where `bracket_content` can be:
- A **continent or region label** → `Africa`, `South Asia`, `Latin America`
- A **sub-national region** → `Bihar`, `California`, `Nairobi`
- An **online indicator** → `Online`, `MTurk`, `online on MTurk`
- **Part of the country name** → `Democratic Republic of the`, `Islamic Republic of`, `Plurinational State of`
- **Noise** → `National`, `all regions`, `Country wide`
- **Empty** → `()`

### Approach
1. Split on `;` to separate multi-country entries
2. For each entry, extract the text before `(` as the raw country string
3. Extract the content inside `()` separately
4. Handle partial country names (where the brackets complete the country name)
5. Clean up the extracted country name

In [5]:
# These are known cases where the bracket content is PART OF the country name
# not additional info. These must be folded back into the country name.
PARTIAL_COUNTRY_NAMES = {
    'Congo': {
        'Democratic Republic of the': 'Democratic Republic of the Congo',
    },
    'Iran': {
        'Islamic Republic of': 'Iran',
    },
    'Bolivia': {
        'Plurinational State of': 'Bolivia',
    },
    'Korea': {
        'Republic of': 'South Korea',
        'Democratic People\'s Republic of': 'North Korea',
    },
    'Venezuela': {
        'Bolivarian Republic of': 'Venezuela',
    },
    'Macedonia': {
        'the former Yugoslav Republic of': 'North Macedonia',
    },
    'Lao': {
        "People's Democratic Republic": "Laos",
    },
    'Tanzania': {
        'United Republic of': 'Tanzania',
    },
    'Moldova': {
        'Republic of': 'Moldova',
    },
    'Palestine': {
        'State of': 'Palestine',
    },
}

def parse_country_entry(entry):
    """
    Parse a single country entry like 'India (Bihar)' or 
    'Congo (Democratic Republic of the) (Kinshasa)'
    
    Returns: (country_name, bracket_content)
    """
    entry = entry.strip()
    
    # Handle Private
    if entry == 'Private':
        return 'Private', ''
    
    # Extract ALL bracket groups
    bracket_groups = re.findall(r'\(([^)]+)\)', entry)
    # Country name = everything before first bracket (or full string if no brackets)
    base = re.sub(r'\(.*', '', entry).strip()
    
    if not bracket_groups:
        return base, ''
    
    # Check if first bracket group is part of the country name
    first_bracket = bracket_groups[0].strip()
    
    # Check partial country name map
    if base in PARTIAL_COUNTRY_NAMES:
        for partial, full_name in PARTIAL_COUNTRY_NAMES[base].items():
            if partial.lower() in first_bracket.lower():
                # This bracket is part of the name - skip it, use remaining brackets
                remaining = bracket_groups[1] if len(bracket_groups) > 1 else ''
                return full_name, remaining.strip()
    
    # Normal case: base is the country, first bracket is detail
    return base, first_bracket.strip()

# Test on tricky cases
test_cases = [
    'Kenya (Nairobi)',
    'Congo (Democratic Republic of the) (Kinshasa)',
    'Iran (Islamic Republic of) (Tehran)',
    'Bolivia (Plurinational State of) (Latin America)',
    'Tanzania, United Republic of ()',
    'Private',
    'United States of America (Online)',
    'India (Bihar - Bhojpur District)',
    'Germany ()',
]

print("=== PARSE TEST ===")
for t in test_cases:
    country, bracket = parse_country_entry(t)
    print(f"  IN:  {t!r}")
    print(f"  OUT: country={country!r}, bracket={bracket!r}")
    print()

=== PARSE TEST ===
  IN:  'Kenya (Nairobi)'
  OUT: country='Kenya', bracket='Nairobi'

  IN:  'Congo (Democratic Republic of the) (Kinshasa)'
  OUT: country='Democratic Republic of the Congo', bracket='Kinshasa'

  IN:  'Iran (Islamic Republic of) (Tehran)'
  OUT: country='Iran', bracket='Tehran'

  IN:  'Bolivia (Plurinational State of) (Latin America)'
  OUT: country='Bolivia', bracket='Latin America'

  IN:  'Tanzania, United Republic of ()'
  OUT: country='Tanzania, United Republic of', bracket=''

  IN:  'Private'
  OUT: country='Private', bracket=''

  IN:  'United States of America (Online)'
  OUT: country='United States of America', bracket='Online'

  IN:  'India (Bihar - Bhojpur District)'
  OUT: country='India', bracket='Bihar - Bhojpur District'

  IN:  'Germany ()'
  OUT: country='Germany', bracket=''



## Step 4 — Explode multi-country entries into separate rows

Some RCTs are conducted across multiple countries. These are stored as semicolon-separated entries in `Country names`:

```
Kenya (); Tanzania, United Republic of ()
```

We split these into **separate rows** — one per country — keeping all other columns identical. This is by design: a multi-country RCT will appear multiple times, distinguished later by `RCT_ID_duplicates`.

In [6]:
rows = []

for _, row in df.iterrows():
    country_names_raw = str(row['Country names']).strip()
    
    # Split on semicolon to get individual country entries
    entries = [e.strip() for e in country_names_raw.split(';') if e.strip()]
    
    for entry in entries:
        country, bracket_content = parse_country_entry(entry)
        new_row = row.to_dict()
        new_row['_parsed_country'] = country
        new_row['_bracket_content'] = bracket_content
        rows.append(new_row)

df_exploded = pd.DataFrame(rows)
print(f"Original rows:  {len(df)}")
print(f"Exploded rows:  {len(df_exploded)}")
print(f"New rows added: {len(df_exploded) - len(df)} (multi-country RCTs)")
print(f"\nSample multi-country exploded:")
sample = df_exploded[df_exploded['RCT_ID'] == 'AEARCTR-0000016'][['RCT_ID', '_parsed_country', '_bracket_content']]
print(sample)

Original rows:  6802
Exploded rows:  7101
New rows added: 299 (multi-country RCTs)

Sample multi-country exploded:
            RCT_ID               _parsed_country _bracket_content
8  AEARCTR-0000016                         Kenya                 
9  AEARCTR-0000016  Tanzania, United Republic of                 


## Step 5 — Standardise country name spellings

The raw data contains spelling variants and informal names that refer to the same country. We normalise these using a lookup dictionary.

Note: `Tanzania, United Republic of` is the UN official name — we keep it as `Tanzania` for readability.

In [9]:
COUNTRY_SPELLING_MAP = {
    # Tanzania variants
    'Tanzania, United Republic of': 'Tanzania',
    # Online/other non-countries
    'nan': None,
    'East, Central, and West Jakarta in Special Capital Region of Jakarta)': 'Indonesia',
    'Online Survey)': None,  # parsing artifact - will be handled by is_online flag
    
    # Spelling variants  
    'Lao': 'Laos',
    "Lao People's Democratic Republic": 'Laos',
    'Viet Nam': 'Vietnam',
    'Russian Federation': 'Russia',
    'Taiwan, Province of China': 'Taiwan',
    'Korea': 'South Korea',
    'Syrian Arab Republic': 'Syria',
    'Hong Kong': 'Hong Kong',
    'Guinea Bissau': 'Guinea-Bissau',
    'Slovak Republic': 'Slovakia',
    
    # Fix any parsing artifacts
    'N/A': None,
}

def standardise_country(name):
    if pd.isna(name) or name == '':
        return None
    name = name.strip()
    return COUNTRY_SPELLING_MAP.get(name, name)

df_exploded['_parsed_country'] = df_exploded['_parsed_country'].apply(standardise_country)

# Check for any None values (parsing artifacts to drop)
nulls = df_exploded['_parsed_country'].isna().sum()
print(f"Rows with unparseable country (to drop): {nulls}")
if nulls > 0:
    print(df_exploded[df_exploded['_parsed_country'].isna()][['Country names', '_parsed_country', '_bracket_content']].head())

df_exploded = df_exploded.dropna(subset=['_parsed_country'])
print(f"Rows after dropping artifacts: {len(df_exploded)}")

print(f"\nUnique countries now: {df_exploded['_parsed_country'].nunique()}")
print(df_exploded['_parsed_country'].value_counts().head(20))

Rows with unparseable country (to drop): 2
     Country names _parsed_country _bracket_content
1440           NaN            None                 
2954           NaN            None                 
Rows after dropping artifacts: 7096

Unique countries now: 135
_parsed_country
Private                                                 4144
United States of America                                 659
India                                                    185
Germany                                                  177
Kenya                                                    110
United Kingdom of Great Britain and Northern Ireland      97
China                                                     88
Bangladesh                                                77
Uganda                                                    75
Pakistan                                                  65
Italy                                                     49
Colombia                                           

In [10]:
print(df_exploded['_parsed_country'].unique())

['Kenya' 'Nigeria' 'Private' 'India' 'Ghana' 'Tanzania' 'Zambia'
 'El Salvador' 'United States of America' 'Bangladesh' 'Sierra Leone'
 'Malawi' 'Cambodia' 'Philippines' 'Ethiopia' 'Uganda' 'Honduras' 'Egypt'
 'Chile' 'Indonesia' 'Mali' 'Colombia' 'Iran' 'Switzerland' 'Peru'
 'South Africa' 'Armenia' 'France' 'Mozambique' 'Pakistan'
 'United Kingdom of Great Britain and Northern Ireland' 'China' 'Mongolia'
 'Bosnia and Herzegovina' 'Mexico' 'Costa Rica' 'Dominican Republic'
 'Rwanda' 'Morocco' 'Democratic Republic of the Congo' 'Niger' 'Turkey'
 'Venezuela' 'Norway' 'Benin' 'Madagascar' 'Jordan' 'Brazil' 'Lesotho'
 'Nepal' 'Nicaragua' 'Cameroon' 'Sweden' 'Senegal' 'Netherlands' 'Italy'
 'Canada' 'Australia' 'Togo' 'Croatia' 'Kosovo' 'Montenegro'
 'North Macedonia' 'Serbia' 'Austria' 'Germany' 'Argentina' 'Liberia'
 'Qatar' 'Jamaica' 'Russia' 'Malaysia' 'Guatemala' 'Bolivia' 'Afghanistan'
 'Spain' 'Denmark' 'Papua New Guinea' 'Taiwan' 'Sri Lanka' 'Vietnam'
 'Japan' 'Solomon Islands' 'Fi

## Step 6 — Detect online / remote RCTs

Some RCTs were conducted online (MTurk, Prolific, etc.). These are flagged via the bracket content. We detect them using keyword matching and create an `is_online` boolean flag.

Online RCTs keep their country (usually US/UK) but are flagged separately.

In [11]:
ONLINE_KEYWORDS = [
    'online', 'mturk', 'prolific', 'm-turk', 'mechanical turk',
    'online experiment', 'online panel', 'online survey', 'online: prolific',
    'web experiment', 'internet', 'crowdsourcing',
]

def is_online_entry(bracket_content):
    if not bracket_content:
        return False
    bc_lower = bracket_content.lower()
    return any(kw in bc_lower for kw in ONLINE_KEYWORDS)

df_exploded['is_online'] = df_exploded['_bracket_content'].apply(is_online_entry)

print(f"Online RCTs: {df_exploded['is_online'].sum()}")
print("\nOnline bracket content examples:")
online_examples = df_exploded[df_exploded['is_online']][['_parsed_country', '_bracket_content']].drop_duplicates()
print(online_examples.head(15))

Online RCTs: 11

Online bracket content examples:
                                        _parsed_country  \
1934                           United States of America   
2365                           United States of America   
2786                           United States of America   
3913  United Kingdom of Great Britain and Northern I...   
4389                                            Germany   
4566                           United States of America   
4792                                             Canada   
4793                           United States of America   
5803                           United States of America   
6087                           United States of America   

                                       _bracket_content  
1934                                              MTurk  
2365                                             Online  
2786                                  Online experiment  
3913                                 (online experiment  
4389  Onli

In [12]:
# Check how many bracket contents contain 'online' but weren't flagged
missed = df_exploded[
    df_exploded['_bracket_content'].str.lower().str.contains('online', na=False) & 
    (df_exploded['is_online'] == False)
]
print(missed[['_parsed_country', '_bracket_content']].drop_duplicates())

Empty DataFrame
Columns: [_parsed_country, _bracket_content]
Index: []


## Step 7 — Map country → continent and continent_region

This is the key fix over the previous notebook. Rather than inferring continent from bracket content (which is inconsistent), we use a **fixed lookup dictionary** that maps every country name to its correct continent and sub-region.

This ensures Afghanistan always maps to `Asia / South Asia`, regardless of whether its bracket content says `(South Asia)`, `(Kabul)`, or `()`.

Sub-regions follow the UN geoscheme where possible.

In [13]:
# country -> (continent, continent_region)
COUNTRY_CONTINENT_MAP = {
    # AFRICA
    'Angola': ('Africa', 'Sub-Saharan Africa'),
    'Benin': ('Africa', 'West Africa'),
    'Botswana': ('Africa', 'Southern Africa'),
    'Burkina Faso': ('Africa', 'West Africa'),
    'Cameroon': ('Africa', 'Central Africa'),
    'Chad': ('Africa', 'Central Africa'),
    'Congo': ('Africa', 'Central Africa'),
    'Democratic Republic of the Congo': ('Africa', 'Central Africa'),
    "Côte d'Ivoire": ('Africa', 'West Africa'),
    'Ethiopia': ('Africa', 'East Africa'),
    'Gambia': ('Africa', 'West Africa'),
    'Ghana': ('Africa', 'West Africa'),
    'Guinea-Bissau': ('Africa', 'West Africa'),
    'Kenya': ('Africa', 'East Africa'),
    'Lesotho': ('Africa', 'Southern Africa'),
    'Liberia': ('Africa', 'West Africa'),
    'Madagascar': ('Africa', 'East Africa'),
    'Malawi': ('Africa', 'East Africa'),
    'Mali': ('Africa', 'West Africa'),
    'Mauritania': ('Africa', 'West Africa'),
    'Morocco': ('Africa', 'North Africa'),
    'Mozambique': ('Africa', 'East Africa'),
    'Namibia': ('Africa', 'Southern Africa'),
    'Niger': ('Africa', 'West Africa'),
    'Nigeria': ('Africa', 'West Africa'),
    'Rwanda': ('Africa', 'East Africa'),
    'Senegal': ('Africa', 'West Africa'),
    'Sierra Leone': ('Africa', 'West Africa'),
    'Somalia': ('Africa', 'East Africa'),
    'South Africa': ('Africa', 'Southern Africa'),
    'South Sudan': ('Africa', 'East Africa'),
    'Tanzania': ('Africa', 'East Africa'),
    'Togo': ('Africa', 'West Africa'),
    'Tunisia': ('Africa', 'North Africa'),
    'Uganda': ('Africa', 'East Africa'),
    'Zambia': ('Africa', 'Southern Africa'),
    'Zimbabwe': ('Africa', 'Southern Africa'),
    'Egypt': ('Africa', 'North Africa'),

    # ASIA
    'Afghanistan': ('Asia', 'South Asia'),
    'Azerbaijan': ('Asia', 'Central Asia'),
    'Bangladesh': ('Asia', 'South Asia'),
    'Bhutan': ('Asia', 'South Asia'),
    'Cambodia': ('Asia', 'Southeast Asia'),
    'China': ('Asia', 'East Asia'),
    'Hong Kong': ('Asia', 'East Asia'),
    'India': ('Asia', 'South Asia'),
    'Indonesia': ('Asia', 'Southeast Asia'),
    'Iran': ('Asia', 'Middle East and North Africa'),
    'Iraq': ('Asia', 'Middle East and North Africa'),
    'Israel': ('Asia', 'Middle East and North Africa'),
    'Japan': ('Asia', 'East Asia'),
    'Jordan': ('Asia', 'Middle East and North Africa'),
    'Kazakhstan': ('Asia', 'Central Asia'),
    'South Korea': ('Asia', 'East Asia'),
    'North Korea': ('Asia', 'East Asia'),
    'Kyrgyzstan': ('Asia', 'Central Asia'),
    'Laos': ('Asia', 'Southeast Asia'),
    'Lebanon': ('Asia', 'Middle East and North Africa'),
    'Malaysia': ('Asia', 'Southeast Asia'),
    'Mongolia': ('Asia', 'East Asia'),
    'Myanmar': ('Asia', 'Southeast Asia'),
    'Nepal': ('Asia', 'South Asia'),
    'Pakistan': ('Asia', 'South Asia'),
    'Philippines': ('Asia', 'Southeast Asia'),
    'Qatar': ('Asia', 'Middle East and North Africa'),
    'Saudi Arabia': ('Asia', 'Middle East and North Africa'),
    'Singapore': ('Asia', 'Southeast Asia'),
    'Sri Lanka': ('Asia', 'South Asia'),
    'Syria': ('Asia', 'Middle East and North Africa'),
    'Taiwan': ('Asia', 'East Asia'),
    'Tajikistan': ('Asia', 'Central Asia'),
    'Thailand': ('Asia', 'Southeast Asia'),
    'Turkey': ('Asia', 'Middle East and North Africa'),
    'Turkmenistan': ('Asia', 'Central Asia'),
    'Uzbekistan': ('Asia', 'Central Asia'),
    'Vietnam': ('Asia', 'Southeast Asia'),
    'Yemen': ('Asia', 'Middle East and North Africa'),
    'Palestine': ('Asia', 'Middle East and North Africa'),

    # EUROPE
    'Albania': ('Europe', 'Southern Europe'),
    'Armenia': ('Europe', 'Eastern Europe'),
    'Austria': ('Europe', 'Western Europe'),
    'Belgium': ('Europe', 'Western Europe'),
    'Bosnia and Herzegovina': ('Europe', 'Eastern Europe'),
    'Bulgaria': ('Europe', 'Eastern Europe'),
    'Croatia': ('Europe', 'Eastern Europe'),
    'Czechia': ('Europe', 'Eastern Europe'),
    'Denmark': ('Europe', 'Northern Europe'),
    'Estonia': ('Europe', 'Northern Europe'),
    'Finland': ('Europe', 'Northern Europe'),
    'France': ('Europe', 'Western Europe'),
    'Georgia': ('Europe', 'Eastern Europe'),
    'Germany': ('Europe', 'Western Europe'),
    'Greece': ('Europe', 'Southern Europe'),
    'Hungary': ('Europe', 'Eastern Europe'),
    'Ireland': ('Europe', 'Northern Europe'),
    'Italy': ('Europe', 'Southern Europe'),
    'Kosovo': ('Europe', 'Eastern Europe'),
    'Latvia': ('Europe', 'Northern Europe'),
    'Lithuania': ('Europe', 'Northern Europe'),
    'Luxembourg': ('Europe', 'Western Europe'),
    'Macedonia': ('Europe', 'Eastern Europe'),
    'North Macedonia': ('Europe', 'Eastern Europe'),
    'Moldova': ('Europe', 'Eastern Europe'),
    'Montenegro': ('Europe', 'Eastern Europe'),
    'Netherlands': ('Europe', 'Western Europe'),
    'Norway': ('Europe', 'Northern Europe'),
    'Poland': ('Europe', 'Eastern Europe'),
    'Portugal': ('Europe', 'Southern Europe'),
    'Romania': ('Europe', 'Eastern Europe'),
    'Russia': ('Europe', 'Eastern Europe'),
    'Serbia': ('Europe', 'Eastern Europe'),
    'Spain': ('Europe', 'Southern Europe'),
    'Sweden': ('Europe', 'Northern Europe'),
    'Switzerland': ('Europe', 'Western Europe'),
    'Ukraine': ('Europe', 'Eastern Europe'),
    'United Kingdom of Great Britain and Northern Ireland': ('Europe', 'Northern Europe'),

    # NORTH AMERICA
    'Canada': ('North America', 'North America'),
    'Guatemala': ('North America', 'Central America'),
    'Honduras': ('North America', 'Central America'),
    'El Salvador': ('North America', 'Central America'),
    'Nicaragua': ('North America', 'Central America'),
    'Costa Rica': ('North America', 'Central America'),
    'Panama': ('North America', 'Central America'),
    'Mexico': ('North America', 'Central America'),
    'United States of America': ('North America', 'North America'),
    'Jamaica': ('North America', 'Caribbean'),
    'Dominican Republic': ('North America', 'Caribbean'),
    'Antigua and Barbuda': ('North America', 'Caribbean'),
    'Saint Lucia': ('North America', 'Caribbean'),
    'Haiti': ('North America', 'Caribbean'),

    # SOUTH AMERICA
    'Argentina': ('South America', 'South America'),
    'Bolivia': ('South America', 'South America'),
    'Brazil': ('South America', 'South America'),
    'Chile': ('South America', 'South America'),
    'Colombia': ('South America', 'South America'),
    'Ecuador': ('South America', 'South America'),
    'Paraguay': ('South America', 'South America'),
    'Peru': ('South America', 'South America'),
    'Uruguay': ('South America', 'South America'),
    'Venezuela': ('South America', 'South America'),

    # OCEANIA
    'Australia': ('Oceania', 'Oceania'),
    'New Zealand': ('Oceania', 'Oceania'),
    'Papua New Guinea': ('Oceania', 'Pacific Islands'),
    'Solomon Islands': ('Oceania', 'Pacific Islands'),
    'Palau': ('Oceania', 'Pacific Islands'),

    # SPECIAL
    'Private': ('Private/Confidential', 'Private/Confidential'),
}

def get_continent(country):
    result = COUNTRY_CONTINENT_MAP.get(country)
    if result:
        return result[0]
    return 'Unknown'

def get_continent_region(country):
    result = COUNTRY_CONTINENT_MAP.get(country)
    if result:
        return result[1]
    return 'Unknown'

df_exploded['continent'] = df_exploded['_parsed_country'].apply(get_continent)
df_exploded['continent_region'] = df_exploded['_parsed_country'].apply(get_continent_region)

# Flag any unknowns so we can fix them
unknown = df_exploded[df_exploded['continent'] == 'Unknown']['_parsed_country'].value_counts()
print(f"Countries not in lookup (Unknown): {len(unknown)}")
if len(unknown) > 0:
    print(unknown)

Countries not in lookup (Unknown): 0


## Step 8 — Extract region_detail

The bracket content, once we've handled continent labels, partial country names, and online flags, often contains a useful sub-national region (state, city, district). We store this as `region_detail`.

We discard generic noise values like `National`, `all regions`, `Country wide` etc.

In [14]:
# Values in bracket content that are continent/region labels (not region_detail)
CONTINENT_LABELS = {
    'africa', 'sub-saharan africa', 'sub saharan africa', 'west africa',
    'east africa', 'north africa', 'southern africa', 'central africa',
    'ssa', 'sub-saharan',
    'south asia', 'southeast asia', 'south-east asia', 'east asia',
    'central asia', 'asia', 'central and east asia',
    'middle east and north africa', 'mena', 'meadle east and north africa',
    'middle east', 'north africa',
    'latin america', 'latin america and the caribbean',
    'latin america & caribbean', 'latin america & carribean',
    'latin american & caribbean', 'lac', 'latin america and caribbean',
    'central america', 'south america', 'south americe', 'caribbean',
    'north america',
    'europe', 'western balkans', 'central, eastern and southeastern europe',
    'central eastern and southeastern europe',
    'western europe', 'eastern europe', 'northern europe', 'southern europe',
    'oceania',
    'pan-african, including 49 countries', 'global',
}

# Noise values that convey no useful location info
NOISE_VALUES = {
    'national', 'nationwide', 'countrywide', 'country-wide', 'country wide',
    'all', 'all regions', 'all of india', 'all of us', 'all the states',
    'all over the nation', 'entire country', 'full country', 'complete country',
    'country', 'multiple', 'multiple locations', 'multiple regions',
    'multiple locations across the u.s.',
    'urban', 'urban areas', 'rural areas, nationwide',
    'national representativeness', 'n/a', 'not applicable', 'other',
    'tc', 'in', 'or', 'ma', 'va', 'nv', 'ny', 'il', 'ca',  # US state abbreviations
    'van', 'wah cantt',  # parsing artifacts
    '(online experiment',  # malformed bracket
}

def extract_region_detail(bracket_content, is_online_flag):
    """
    Returns the region_detail string, or 'No additional info' if not useful.
    """
    if not bracket_content or pd.isna(bracket_content):
        return 'No additional info'
    
    bc = bracket_content.strip()
    bc_lower = bc.lower()
    
    # Skip continent labels
    if bc_lower in CONTINENT_LABELS:
        return 'No additional info'
    
    # Skip noise
    if bc_lower in NOISE_VALUES:
        return 'No additional info'
    
    # Skip online indicators (already captured by is_online flag)
    if is_online_flag:
        return 'Online/Remote'
    
    # Skip if it looks like a continent label with minor variation
    for label in CONTINENT_LABELS:
        if bc_lower == label:
            return 'No additional info'
    
    return bc

df_exploded['region_detail'] = df_exploded.apply(
    lambda row: extract_region_detail(row['_bracket_content'], row['is_online']),
    axis=1
)

print("region_detail value counts (top 20):")
print(df_exploded['region_detail'].value_counts().head(20))

region_detail value counts (top 20):
region_detail
No additional info    6049
California              27
Nairobi                 17
Punjab                  12
Bihar                   12
New York                11
Texas                   10
Online/Remote           10
Beijing                  9
Illinois                 9
Karnataka                9
Dhaka                    8
West Bengal              8
England                  8
Massachusetts            7
Hamburg                  7
Rajasthan                6
Uttar Pradesh            6
New York City            6
Tamil Nadu               6
Name: count, dtype: int64


## Step 9 — Build country/region flags

Now that each row has a clean country, we compute:
- `country_count`: how many countries this RCT spans
- `region_count`: how many distinct sub-national regions this RCT spans within a country
- `is_multi_country`: True if country_count > 1
- `is_multi_region`: True if region_count > 1 for the same RCT within the same country

In [15]:
# country_count: number of distinct countries per RCT_ID
country_counts = df_exploded.groupby('RCT_ID')['_parsed_country'].nunique().rename('country_count')
df_exploded = df_exploded.merge(country_counts, on='RCT_ID', how='left')

# region_count: number of distinct region_details per RCT_ID + country combo
region_counts = (
    df_exploded[df_exploded['region_detail'] != 'No additional info']
    .groupby(['RCT_ID', '_parsed_country'])['region_detail']
    .nunique()
    .rename('region_count')
    .reset_index()
)
df_exploded = df_exploded.merge(region_counts, on=['RCT_ID', '_parsed_country'], how='left')
df_exploded['region_count'] = df_exploded['region_count'].fillna(0).astype(int)

# Flags
df_exploded['is_multi_country'] = df_exploded['country_count'] > 1
df_exploded['is_multi_region']  = df_exploded['region_count'] > 1

print("country_count distribution:")
print(df_exploded['country_count'].value_counts().sort_index())
print("\nregion_count distribution:")
print(df_exploded['region_count'].value_counts().sort_index())
print(f"\nMulti-country RCTs: {df_exploded['is_multi_country'].sum()}")
print(f"Multi-region RCTs:  {df_exploded['is_multi_region'].sum()}")

country_count distribution:
country_count
1     6765
2      114
3       57
4       16
5       10
6       30
7       14
9        9
10      20
11      11
14      14
18      36
Name: count, dtype: int64

region_count distribution:
region_count
0    6046
1     979
2      32
3      12
4      12
6       6
9       9
Name: count, dtype: int64

Multi-country RCTs: 331
Multi-region RCTs:  71


In [16]:
# What are the RCTs with very high country counts?
high = df_exploded[df_exploded['country_count'] >= 10][['RCT_ID', 'country_count', 'Country names']].drop_duplicates('RCT_ID')
print(high[['RCT_ID', 'country_count', 'Country names']].to_string())

               RCT_ID  country_count                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             Country names
1544  AEARCTR-0002534             10                                                                                                                                                                                                                                                                                                                                                                     

## Step 10 — Clean Keywords

The `Keywords` column suffers from double-escaping that occurred during CSV serialisation. Raw values look like:

```
[\\electoral\\", \\"public goods\\", ...]
```

We strip the escaping and produce a clean pipe-separated string:
```
electoral|public goods|corruption|audits
```

Pipe-separated (rather than comma) avoids ambiguity with keywords that contain commas.

In [19]:
def clean_keywords(raw):
    if pd.isna(raw) or raw == '':
        return ''
    
    s = str(raw)
    # Remove all backslashes
    s = s.replace('\\', '')
    # Remove surrounding [  and trailing ]" or ]
    s = re.sub(r'^\[|\]"?$', '', s).strip()
    # Now looks like: electoral", "public goods", "corruption"
    # Split on ", " to get individual terms
    terms = [t.strip().strip('"') for t in s.split('",')]
    # Clean each term
    seen = set()
    result = []
    for t in terms:
        t_clean = t.strip().strip('"').strip().lower()
        if t_clean and t_clean not in seen:
            seen.add(t_clean)
            result.append(t_clean)
    return '|'.join(result)

df_exploded['keywords_clean'] = df_exploded['Keywords'].apply(clean_keywords)

print("Keyword cleaning samples:")
for raw, clean in zip(df_exploded['Keywords'].head(5), df_exploded['keywords_clean'].head(5)):
    print(f"  RAW:   {raw[:80]}")
    print(f"  CLEAN: {clean}")
    print()

# Top keywords across all RCTs
all_kw = []
for kws in df_exploded['keywords_clean']:
    if kws:
        all_kw.extend(kws.split('|'))
kw_counts = Counter(all_kw)
print(f"Total unique keywords: {len(kw_counts)}")
print("\nTop 20 keywords:")
for kw, count in kw_counts.most_common(20):
    print(f"  {kw}: {count}")

Keyword cleaning samples:
  RAW:   [\electoral\", \"public goods\", \"corruption\", \"audits\", \"voting\", \"accou
  CLEAN: electoral|public goods|corruption|audits|voting|accountability|information

  RAW:   [\health\", \"pregnancy\", \"maternal mortality\", \"neonatal mortality\", \"com
  CLEAN: health|pregnancy|maternal mortality|neonatal mortality|community intervention|voluntary health worker|safe birth kit|maternal child health

  RAW:   [\education\", \"remedial education\"]"
  CLEAN: education|remedial education

  RAW:   [\governance\", \"employment\", \"welfare\"]"
  CLEAN: governance|employment|welfare

  RAW:   [\health\", \"Nutrition\", \"Anemia\"]"
  CLEAN: health|nutrition|anemia

Total unique keywords: 6900

Top 20 keywords:
  behavior: 1730
  education: 1618
  labor: 1400
  health: 1262
  welfare: 986
  other: 894
  finance: 881
  gender: 822
  governance: 797
  firms_and_productivity: 565
  environment_and_energy: 557
  agriculture: 455
  electoral: 295
  lab: 291
  

In [20]:
# Print exact repr of first 3 keyword values
for k in df_exploded['Keywords'].head(3):
    print(repr(k))
    print()

'[\\electoral\\", \\"public goods\\", \\"corruption\\", \\"audits\\", \\"voting\\", \\"accountability\\", \\"information\\"]"'

'[\\health\\", \\"pregnancy\\", \\"maternal mortality\\", \\"neonatal mortality\\", \\"community intervention\\", \\"voluntary health worker\\", \\"safe birth kit\\", \\"maternal child health\\"]"'

'[\\education\\", \\"remedial education\\"]"'



## Step 11 — Create RCT_ID_duplicates

We need a **unique row-level identifier** because multi-country RCTs now occupy multiple rows with the same `RCT_ID`.

Rules:
- If an `RCT_ID` appears only once → `RCT_ID_duplicates` = same as `RCT_ID`
- If an `RCT_ID` appears multiple times → append `_1`, `_2`, `_3` etc.

In [21]:
# Use cumcount to number rows within each RCT_ID group
df_exploded['_row_num'] = df_exploded.groupby('RCT_ID').cumcount() + 1
df_exploded['_total_rows'] = df_exploded.groupby('RCT_ID')['RCT_ID'].transform('count')

df_exploded['RCT_ID_duplicates'] = df_exploded.apply(
    lambda row: row['RCT_ID'] if row['_total_rows'] == 1
                else f"{row['RCT_ID']}_{row['_row_num']}",
    axis=1
)

# Validate uniqueness
print(f"Total rows:                  {len(df_exploded)}")
print(f"Unique RCT_ID_duplicates:    {df_exploded['RCT_ID_duplicates'].nunique()}")
print(f"Every row is unique:         {len(df_exploded) == df_exploded['RCT_ID_duplicates'].nunique()}")

print("\nSample multi-country example:")
ex = df_exploded[df_exploded['_total_rows'] > 1][['RCT_ID', 'RCT_ID_duplicates', '_parsed_country']].head(6)
print(ex)

Total rows:                  7096
Unique RCT_ID_duplicates:    7096
Every row is unique:         True

Sample multi-country example:
             RCT_ID  RCT_ID_duplicates           _parsed_country
8   AEARCTR-0000016  AEARCTR-0000016_1                     Kenya
9   AEARCTR-0000016  AEARCTR-0000016_2                  Tanzania
15  AEARCTR-0000024  AEARCTR-0000024_1               El Salvador
16  AEARCTR-0000024  AEARCTR-0000024_2  United States of America
17  AEARCTR-0000025  AEARCTR-0000025_1                Bangladesh
18  AEARCTR-0000025  AEARCTR-0000025_2                Bangladesh


## Step 12 — Assemble final dataframe

We select and rename columns into their final structure, dropping the intermediate `_` columns used during processing.

In [25]:
df_final = df_exploded[[
    'RCT_ID',
    'RCT_ID_duplicates',
    'Title',
    'Url',
    'Status',
    'Start date',
    'End date',
    'First registered on',
    'Last update date',
    'Published at',
    'Primary Investigator',
    'DOI Number',
    'Keywords',
    'keywords_clean',
    'Country names',
    '_parsed_country',
    'region_detail',
    'continent',
    'continent_region',
    'country_count',
    'region_count',
    'is_multi_country',
    'is_multi_region',
    'is_online',
]].rename(columns={'_parsed_country': 'country'})

# Reorder so identifiers come first
cols_order = [
    'RCT_ID', 'RCT_ID_duplicates', 'country', 'region_detail',
    'continent', 'continent_region',
    'country_count', 'region_count', 'is_multi_country', 'is_multi_region', 'is_online',
    'Title', 'Status', 'Start date', 'End date', 'First registered on',
    'Last update date', 'Published at', 'Primary Investigator',
    'DOI Number', 'Keywords', 'keywords_clean', 'Country names', 'Url',
]
df_final = df_final[cols_order]

print(f"Final shape: {df_final.shape}")
print(f"\nColumns: {df_final.columns.tolist()}")
df_final.head(3)

Final shape: (7096, 24)

Columns: ['RCT_ID', 'RCT_ID_duplicates', 'country', 'region_detail', 'continent', 'continent_region', 'country_count', 'region_count', 'is_multi_country', 'is_multi_region', 'is_online', 'Title', 'Status', 'Start date', 'End date', 'First registered on', 'Last update date', 'Published at', 'Primary Investigator', 'DOI Number', 'Keywords', 'keywords_clean', 'Country names', 'Url']


,RCT_ID,RCT_ID_duplicates,country,region_detail,continent,continent_region,country_count,region_count,is_multi_country,is_multi_region,...,End date,First registered on,Last update date,Published at,Primary Investigator,DOI Number,Keywords,keywords_clean,Country names,Url
0,AEARCTR-0000005,AEARCTR-0000005,Kenya,No additional info,Africa,East Africa,1,0,False,False,...,2014-05-31,2013-05-21,2017-05-02,2017-05-02 16:17:17 -0400,Kelly Zhang kwzhang@mit.edu,10.1257/rct.5-8.0,"[\electoral\"", \""public goods\"", \""corruption\...",electoral|public goods|corruption|audits|votin...,Kenya (National),http://www.socialscienceregistry.org/trials/5
1,AEARCTR-0000006,AEARCTR-0000006,Nigeria,No additional info,Africa,West Africa,1,0,False,False,...,2015-12-31,2013-05-26,2016-09-07,2016-09-07 17:44:46 -0400,Jessica Leight j.leight@cgiar.org,10.1257/rct.6-2.0,"[\health\"", \""pregnancy\"", \""maternal mortalit...",health|pregnancy|maternal mortality|neonatal m...,Nigeria (Africa),http://www.socialscienceregistry.org/trials/6
2,AEARCTR-0000008,AEARCTR-0000008,Private,No additional info,Private/Confidential,Private/Confidential,1,0,False,False,...,2014-12-31,2013-06-15,2013-06-15,2013-06-15 09:00:16 -0400,Esther Duflo eduflo@mit.edu,10.1257/rct.8-1.0,"[\education\"", \""remedial education\""]""",education|remedial education,Private,http://www.socialscienceregistry.org/trials/8


## Step 13 — Validation

Before saving, we run a series of checks to confirm the data is clean and consistent.

In [26]:
print("=" * 60)
print("VALIDATION REPORT")
print("=" * 60)

# 1. Row uniqueness
assert df_final['RCT_ID_duplicates'].nunique() == len(df_final), \
    "FAIL: RCT_ID_duplicates is not unique!"
print("✓ RCT_ID_duplicates: every row is unique")

# 2. No nulls in key columns
key_cols = ['RCT_ID', 'RCT_ID_duplicates', 'country', 'continent', 'continent_region']
for col in key_cols:
    nulls = df_final[col].isna().sum()
    assert nulls == 0, f"FAIL: {col} has {nulls} null values!"
    print(f"✓ {col}: no nulls")

# 3. No Unknown continents
unknowns = (df_final['continent'] == 'Unknown').sum()
print(f"\n{'✓' if unknowns == 0 else '✗'} Unknown continents: {unknowns}")
if unknowns > 0:
    print("  Countries with Unknown continent:")
    print(df_final[df_final['continent'] == 'Unknown']['country'].value_counts())

# 4. Country → continent consistency (each country maps to exactly 1 continent)
country_continent = df_final[df_final['country'] != 'Private'].groupby('country')['continent'].nunique()
inconsistent = country_continent[country_continent > 1]
print(f"\n{'✓' if len(inconsistent) == 0 else '✗'} Countries with inconsistent continent mapping: {len(inconsistent)}")
if len(inconsistent) > 0:
    print(inconsistent)

# 5. Multi-country flag consistency
multi_check = df_final[df_final['is_multi_country'] == True]['country_count'].min()
print(f"\n✓ Min country_count where is_multi_country=True: {multi_check} (should be ≥ 2)")

# 6. Keywords clean check
empty_kw = (df_final['keywords_clean'] == '').sum()
print(f"\nRCTs with empty keywords_clean: {empty_kw}")

# 7. Summary stats
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Total rows:            {len(df_final)}")
print(f"Unique RCT_IDs:        {df_final['RCT_ID'].nunique()}")
print(f"Private RCTs:          {(df_final['country'] == 'Private').sum()}")
print(f"Multi-country RCTs:    {df_final['is_multi_country'].sum()}")
print(f"Online RCTs:           {df_final['is_online'].sum()}")
print(f"\nContinent breakdown:")
print(df_final['continent'].value_counts())
print(f"\nStatus breakdown:")
print(df_final['Status'].value_counts())

VALIDATION REPORT
✓ RCT_ID_duplicates: every row is unique
✓ RCT_ID: no nulls
✓ RCT_ID_duplicates: no nulls
✓ country: no nulls
✓ continent: no nulls
✓ continent_region: no nulls

✓ Unknown continents: 0

✓ Countries with inconsistent continent mapping: 0

✓ Min country_count where is_multi_country=True: 2 (should be ≥ 2)

RCTs with empty keywords_clean: 0

SUMMARY
Total rows:            7096
Unique RCT_IDs:        6800
Private RCTs:          4144
Multi-country RCTs:    331
Online RCTs:           11

Continent breakdown:
continent
Private/Confidential    4144
North America            766
Europe                   672
Asia                     668
Africa                   604
South America            190
Oceania                   52
Name: count, dtype: int64

Status breakdown:
Status
in_development    2657
completed         2205
on_going          2186
withdrawn           46
abandoned            2
Name: count, dtype: int64


## Step 14 — Save outputs

We save two files:
1. `rct_cleaned_data.csv` — the main analysis-ready dataset
2. `comparison_original_vs_cleaned.csv` — a lightweight reference file mapping `Country names` → parsed columns (useful for QA and debugging)

In [27]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_final.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}  ({len(df_final)} rows)")

# Comparison / QA file
df_comp = df_final[[
    'Country names', 'RCT_ID', 'RCT_ID_duplicates',
    'country', 'region_detail', 'continent', 'continent_region',
    'country_count', 'region_count', 'is_multi_country', 'is_multi_region', 'is_online'
]]
df_comp.to_csv(COMP_PATH, index=False)
print(f"Saved: {COMP_PATH}  ({len(df_comp)} rows)")

Saved: ../data/processed/rct_cleaned_data.csv  (7096 rows)
Saved: ../data/processed/comparison_original_vs_cleaned.csv  (7096 rows)
